# Scientific Reports successor — Step 2 bootstrap and closure

This notebook mounts Google Drive, installs the frozen parent and successor packages, validates the parent-source chain, performs an actual storage round-trip test, writes and independently reopens the Step 2 artifact set, and stops. **No scientific calculations are authorized or executed in this notebook.**

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

REPOSITORY = "https://github.com/khalid-saqr/picoNewton.git"
BRANCH = "successor/scirep-waveform-susceptibility"
COLAB_REPO_ROOT = Path("/content/picoNewton")
DRIVE_SUBDIR = "MyDrive/picoNewton_susceptibility"
IN_COLAB = "google.colab" in sys.modules
print({"in_colab": IN_COLAB, "branch": BRANCH})

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
else:
    print("Local execution: Google Drive mount skipped.")

In [ ]:
if IN_COLAB:
    if not (COLAB_REPO_ROOT / ".git").exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY, str(COLAB_REPO_ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(COLAB_REPO_ROOT), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(COLAB_REPO_ROOT), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(COLAB_REPO_ROOT), "reset", "--hard", f"origin/{BRANCH}"], check=True)
    REPO_ROOT = COLAB_REPO_ROOT
else:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    REPO_ROOT = next((root for root in candidates if (root / "picoNewton_v3").is_dir()), None)
    if REPO_ROOT is None:
        raise FileNotFoundError("Run locally from inside the picoNewton repository.")
print("Repository root:", REPO_ROOT)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT / "picoNewton_v3")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT / "piconewton_susceptibility")], check=True)

In [ ]:
from piconewton_susceptibility.bootstrap import BootstrapConfig, bootstrap_environment
from piconewton_susceptibility.validation import validate_bootstrap_artifacts

result = bootstrap_environment(
    BootstrapConfig(
        repo_root=REPO_ROOT,
        storage_mode="drive" if IN_COLAB else "local",
        drive_subdir=DRIVE_SUBDIR,
        local_root=REPO_ROOT / "piconewton_susceptibility_outputs",
    )
)
print(json.dumps({k: v for k, v in result.items() if k != "validation"}, indent=2, sort_keys=True))

In [ ]:
manifest = result["manifest"]
gate = result["completion_gate"]
validation = validate_bootstrap_artifacts(
    result["bootstrap_root"],
    require_claim_bearing=True,
    expected_storage_mode="drive" if IN_COLAB else "local",
)

assert manifest["scientific_calculations_run"] is False
assert manifest["scientific_calculations_authorized"] is False
assert manifest["parent_source_validation_passed"] is True
assert manifest["runtime_validation_passed"] is True
assert gate["passed"] is True
assert gate["allowed_next_step"] == 3
assert validation["passed"] is True
if IN_COLAB:
    assert str(Path(result["output_root"])).startswith("/content/drive/")
print(json.dumps(validation["checks"], indent=2, sort_keys=True))
print("Step 2 is closed. Step 3 has not started.")

## Stop boundary

The notebook intentionally ends here. The completion gate authorizes Step 3 as the next workflow stage only after parent-source validation, storage round-trip verification, final checksum closure, and independent artifact reopening all pass. Scientific calculations remain prohibited inside this Step 2 notebook.